In [ ]:
import actionet
import scanpy as sc
import anndata
import pandas as pd
import numpy as np

import lets_plot
lets_plot.LetsPlot.setup_html(no_js=True)

In [ ]:
adata_src = anndata.read_h5ad("../data/test_adata.h5ad", backed = "r")
adata_src.copy(filename="../data/adata_working_test.h5ad")
adata_src.file.close()

In [ ]:
adata_mem = anndata.read_h5ad("../data/adata_working_test.h5ad")
adata_op = anndata.read_h5ad("../data/adata_working_test.h5ad", backed = "r+")

In [ ]:
actionet.filter_anndata(adata_mem, min_cells_per_feat=0.01, inplace=True)
actionet.normalize_anndata(adata_mem, target_sum = 1e6, log_base=2, log_transform=True, inplace=True, dtype_out="float32")

actionet.filter_anndata(adata_op, min_cells_per_feat=0.01, inplace=True)
actionet.normalize_anndata(adata_op, target_sum = 1e6, log_base=2, log_transform=True, inplace=True, dtype_out="float64")

In [ ]:
adata_mem.X
# adata_op.X

In [ ]:
adata_mem.X.sum() - adata_op.X.to_memory().sum()

In [ ]:
import math
math.isclose(adata_mem.X.astype(np.float32).sum(),adata_op.X.to_memory().astype(np.float32).sum())

In [ ]:
adata_mem.X = adata_mem.X.astype(np.float32)
# adata_op.X

In [ ]:
adata_mem.X.sum() - adata_op.X.to_memory().sum()

In [ ]:
actionet.reduce_kernel(adata_mem, n_components=30, key_added='action', inplace=True, backed_n_threads=8, svd_algorithm="halko")
actionet.reduce_kernel(adata_op, n_components=30, key_added='action', inplace=True, backed_n_threads=8, svd_algorithm="halko")

In [ ]:
print(adata_mem.obsm['action'].sum())
adata_op.obsm['action'].sum()

In [ ]:
actionet.correct_batch_effect(adata, batch_key="UID", inplace=True)

In [ ]:
actionet.run_actionet(adata_mem, compute_specificity_parallel=False, inplace=True, layout_epochs=200, reduction_key="action")
actionet.run_actionet(adata_op, compute_specificity_parallel=False, inplace=True, layout_epochs=200, reduction_key="action")

In [ ]:
actionet.plot_umap(adata_mem, color='CellLabel', basis='umap_2d_actionet')

In [ ]:
actionet.plot_umap(adata_op, color='CellLabel', basis='umap_2d_actionet')

In [ ]:
markers = actionet.find_markers(adata, adata.obs['CellLabel'], features_use="Gene", top_genes=30, return_type='dataframe')

In [ ]:
annots_out = actionet.annotate_cells(adata, markers, layer = None, method='vision', features_use='Gene', use_enrichment=True, use_lpa=False, n_threads=0)

In [ ]:
actionet.plot_umap(adata, color=annots_out['labels'], basis='umap_2d_actionet')